## Add name of dashboard to each csv 

In [6]:
import os
import pandas as pd

# Folder with all your updated CSV files
folder_path = "dashboards/updated_data"

for filename in os.listdir(folder_path):
    if filename.endswith(".csv") and filename.startswith("updated_"):
        # Extract dashboard ID (e.g. "510" from "updated_510 Dashboard.csv")
        dashboard_id = filename.replace("updated_", "").split()[0]

        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Add or update the 'contest' column
        df["contest"] = dashboard_id  # overwrite if it already exists

        # Move 'contest' to the first column
        cols = df.columns.tolist()
        if cols[0] != "contest":
            cols = ["contest"] + [col for col in cols if col != "contest"]
            df = df[cols]

        # Save back
        df.to_csv(file_path, index=False)
        #print(f"✅ Processed: {filename} (contest = {dashboard_id})")


## Combine all of dashboards 

In [7]:
import os
import pandas as pd

# Folder with all your CSVs
folder_path = "dashboards/updated_data"

# List to hold each DataFrame
all_dfs = []

# Loop through files and append to list
for filename in os.listdir(folder_path):
    if filename.endswith(".csv") and filename.startswith("updated_"):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)
        all_dfs.append(df)

# Combine into a single DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)

# Save to a new file
combined_df.to_csv("all_updated_captions.csv", index=False)
print("✅ Combined CSV saved as 'all_updated_captions.csv'")


✅ Combined CSV saved as 'all_updated_captions.csv'


In [8]:
df=pd.read_csv('all_updated_captions.csv')
df.head(10)

/var/folders/s2/1r_lbpts7pvcd7p7h17w62tm0000gn/T/ipykernel_58455/3870191993.py:1: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('all_updated_captions.csv')


,contest,rank,caption,mean,precision,votes,not_funny,somewhat_funny,funny,has_question_mark,ends_with_punctuation,caption_length
0,615,0.0,"I can hear the wife now: ""You caught it, you c...",1.932124,0.010811,5569,2014,1919,1636,False,False,55.0
1,615,1.0,Must be another Trump hotel that's underwater,1.885976,0.020245,1640,655,517,468,False,False,45.0
2,615,2.0,I've finally caught something my wife will let...,1.885218,0.010723,5358,1993,1987,1378,False,True,74.0
3,615,3.0,"""I hate fishing in the Hamptons"".",1.880827,0.010770,5513,2130,1910,1473,False,True,33.0
4,615,4.0,Too bad it's catch and release. My wife would ...,1.859643,0.010407,5600,2146,2094,1360,False,True,56.0
5,615,5.0,We must be at Pier 1.,1.855731,0.010731,5566,2243,1883,1440,False,True,21.0
6,615,6.0,You're gonna need a nicer boat.,1.843970,0.011878,4544,1866,1521,1157,False,True,31.0
7,615,7.0,"Finally, something my wife will let me mount i...",1.837434,0.010478,5487,2171,2037,1279,False,True,64.0
8,615,8.0,Those fish are having a ball down there.,1.837133,0.013611,3334,1345,1187,802,False,True,40.0
9,615,9.0,Now I wish I hadn't thrown back that Chippenda...,1.823428,0.018685,1733,704,631,398,False,True,55.0


In [9]:
df = pd.read_csv("all_updated_captions.csv")

df["model"] = "Human"
df.to_csv("all_updated_captions.csv", index=False)
print(" Added 'model' column with value 'Human'")


/var/folders/s2/1r_lbpts7pvcd7p7h17w62tm0000gn/T/ipykernel_58455/1153709775.py:1: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("all_updated_captions.csv")


 Added 'model' column with value 'Human'


## Combine only top 5 and bottom 5s 

In [18]:
import os
import pandas as pd

folder_path = "dashboards/top_captions"

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # ✅ Clean 'contest' column: remove ' Dashboard'
        df["contest"] = df["contest"].astype(str).str.replace(" Dashboard", "", regex=False)

        # ✅ Add NLP feature columns
        df["has_question_mark"] = df["caption"].astype(str).str.contains(r"\?")
        df["ends_with_punctuation"] = df["caption"].astype(str).str.extract(r"([\.\!\?])$").notnull()
        df["caption_length"] = df["caption"].astype(str).str.strip().str.len()

        # ✅ Add 'model' column with label 'Human'
        df["model"] = "Human"

        # ✅ Save back
        df.to_csv(file_path, index=False)
        #print(f"✅ Processed: {filename}")


In [ ]:
import os
import pandas as pd

folder_path = "dashboards/top_captions"
combined = []

# Read and collect all CSVs
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)
        combined.append(df)

# Combine them into one DataFrame
merged_df = pd.concat(combined, ignore_index=True)

# 🔢 Convert contest to numeric (just in case it's a string)
merged_df["contest"] = pd.to_numeric(merged_df["contest"], errors="coerce")

# 📈 Sort by contest number
merged_df = merged_df.sort_values(by="contest").reset_index(drop=True)

# Save sorted version
merged_df.to_csv("all_top_captions_sorted.csv", index=False)
print("✅ Saved sorted file as 'all_top_captions_sorted.csv'")


## AI models and top captions wide format 

In [16]:
import pandas as pd

# Load files
human_df = pd.read_csv("all_top_captions.csv")
chatgpt_df = pd.read_csv("ai_captions_chatGPT_enriched.csv")
claude_df = pd.read_csv("claude_captions_enriched.csv")

# Select only useful columns and rename for clarity
human_df = human_df[[
    "contest", "caption", "model", "has_question_mark", "ends_with_punctuation", "caption_length",
    "funny_votes", "unfunny_votes", "votes", "category"
]].rename(columns={
    "caption": "caption_human",
    "model": "model_human",
    "has_question_mark": "has_question_mark_human",
    "ends_with_punctuation": "ends_with_punctuation_human",
    "caption_length": "caption_length_human"
})

chatgpt_df = chatgpt_df[[
    "contest", "caption", "model", "has_question_mark", "ends_with_punctuation", "caption_length"
]].rename(columns={
    "caption": "caption_chatgpt",
    "model": "model_chatgpt",
    "has_question_mark": "has_question_mark_chatgpt",
    "ends_with_punctuation": "ends_with_punctuation_chatgpt",
    "caption_length": "caption_length_chatgpt"
})

claude_df = claude_df[[
    "contest", "caption", "model", "has_question_mark", "ends_with_punctuation", "caption_length"
]].rename(columns={
    "caption": "caption_claude",
    "model": "model_claude",
    "has_question_mark": "has_question_mark_claude",
    "ends_with_punctuation": "ends_with_punctuation_claude",
    "caption_length": "caption_length_claude"
})

# Merge on 'contest'
merged = pd.merge(human_df, chatgpt_df, on="contest", how="outer")
merged = pd.merge(merged, claude_df, on="contest", how="outer")

# Save to final file
merged.to_csv("combined_captions_detailed.csv", index=False)
print("✅ Merged and saved as 'combined_captions_detailed.csv'")


✅ Merged and saved as 'combined_captions_detailed.csv'


In [17]:
df=pd.read_csv('combined_captions_comparison.csv')
df.head(10)

,contest,caption_human,model_human,caption_chatgpt,model_chatgpt,caption_claude,model_claude
0,510,I'm a congressman--obstruction is my job.,Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
1,510,Does this suit make me look flat?,Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
2,510,"I used to lie in the gutter, but then I quit d...",Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
3,510,"I'm what they mean when they say, 'The middle ...",Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
4,510,"If you can meditate here, you can meditate any...",Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
5,510,I'm a congressman--obstruction is my job.,Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
6,510,Does this suit make me look flat?,Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
7,510,"I used to lie in the gutter, but then I quit d...",Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
8,510,"If you can meditate here, you can meditate any...",Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude
9,510,I've decided to take it lying down.,Human,Just another day in the Parallel Parking World...,ChatGPT (gpt-4o),"Having a blast, honey? Maybe we should try sep...",Claude


## AI models and top captions long format 

In [19]:
import pandas as pd

# Load your combined wide-format CSV
df = pd.read_csv("combined_captions_detailed.csv")

# Build separate DataFrames for each model
human = df[[
    "contest", "caption_human", "model_human",
    "has_question_mark_human", "ends_with_punctuation_human", "caption_length_human",
    "funny_votes", "unfunny_votes", "votes", "category"
]].rename(columns={
    "caption_human": "caption",
    "model_human": "model",
    "has_question_mark_human": "has_question_mark",
    "ends_with_punctuation_human": "ends_with_punctuation",
    "caption_length_human": "caption_length"
})

chatgpt = df[[
    "contest", "caption_chatgpt", "model_chatgpt",
    "has_question_mark_chatgpt", "ends_with_punctuation_chatgpt", "caption_length_chatgpt"
]].rename(columns={
    "caption_chatgpt": "caption",
    "model_chatgpt": "model",
    "has_question_mark_chatgpt": "has_question_mark",
    "ends_with_punctuation_chatgpt": "ends_with_punctuation",
    "caption_length_chatgpt": "caption_length"
})
# Fill missing human-only columns with NaN
chatgpt["funny_votes"] = None
chatgpt["unfunny_votes"] = None
chatgpt["votes"] = None
chatgpt["category"] = None

claude = df[[
    "contest", "caption_claude", "model_claude",
    "has_question_mark_claude", "ends_with_punctuation_claude", "caption_length_claude"
]].rename(columns={
    "caption_claude": "caption",
    "model_claude": "model",
    "has_question_mark_claude": "has_question_mark",
    "ends_with_punctuation_claude": "ends_with_punctuation",
    "caption_length_claude": "caption_length"
})
claude["funny_votes"] = None
claude["unfunny_votes"] = None
claude["votes"] = None
claude["category"] = None

# Combine into long format
long_df = pd.concat([human, chatgpt, claude], ignore_index=True)

# Save it
long_df.to_csv("captions_long_format.csv", index=False)
print("✅ Saved long-format file as 'captions_long_format.csv'")


✅ Saved long-format file as 'captions_long_format.csv'


# combine with semantic analysis 

In [25]:
import pandas as pd
import os

base_dir = "new_yorker_contest"
csv_path = os.path.join("semantic_similarity_with_matched_category.csv")
svg_folder = os.path.join("similarity_values")
output_path = os.path.join("semantic_similarity_with_svg_path.csv")

# Load CSV
df = pd.read_csv(csv_path)

# Get all SVG filenames and map them to contest numbers
svg_map = {
    int(os.path.splitext(f)[0]): os.path.join(svg_folder, f)
    for f in os.listdir(svg_folder)
    if f.endswith(".svg") and os.path.splitext(f)[0].isdigit()
}

# Add SVG path to DataFrame based on contest number
df["svg_path"] = df["contest"].map(svg_map)

# Save the updated CSV
df.to_csv(output_path, index=False)
print(f"✅ CSV updated with SVG paths and saved to:\n{output_path}")


✅ CSV updated with SVG paths and saved to:
semantic_similarity_with_svg_path.csv


## combine svg, theme + remove Claude's extra words. 

In [27]:
import pandas as pd
import re

# File paths (edit if yours differ)
themes_path = "semantic_similarity_with_clean_themes.csv"
svg_path = "semantic_similarity_with_svg_path.csv"
output_path = "semantic_similarity_combined_final.csv"

# Load both CSVs
df_themes = pd.read_csv(themes_path)
df_svg = pd.read_csv(svg_path)

# Merge on 'contest' and 'ai_caption'
df_merged = pd.merge(
    df_themes,
    df_svg[["contest", "ai_caption", "svg_path"]],
    on=["contest", "ai_caption"],
    how="left"
)

# Clean Claude captions: strip everything after "The image"
def clean_claude_caption(row):
    if row["ai_model"] == "Claude":
        return re.split(r"\bThe image\b", str(row["ai_caption"]))[0].strip()
    return row["ai_caption"]

df_merged["ai_caption"] = df_merged.apply(clean_claude_caption, axis=1)

# Save result
df_merged.to_csv(output_path, index=False)
print(f"✅ Combined file saved at:\n{output_path}")


✅ Combined file saved at:
semantic_similarity_combined_final.csv


In [30]:
import pandas as pd

# Load your CSV
file_path = "semantic_similarity_with_svg_path.csv"
df = pd.read_csv(file_path)

# Define a function to clean Claude captions
def clean_claude_caption(caption):
    if pd.isna(caption):
        return caption
    return caption.split(" The")[0].strip()  # Remove everything from " The"

# Apply only to Claude rows
df.loc[df["ai_model"] == "Claude", "ai_caption"] = df.loc[df["ai_model"] == "Claude", "ai_caption"].apply(clean_claude_caption)

# Save the cleaned file
output_path = "semantic_similarity_with_svg_path.csv"
df.to_csv(output_path, index=False)

print(f"✅ Cleaned file saved to: {output_path}")


✅ Cleaned file saved to: semantic_similarity_with_svg_path.csv
